In [ ]:
import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
df = pd.read_csv("processed_data/merged_data.csv")

print(df.shape)

df.head()

In [ ]:
df = df.drop_duplicates(subset=["song_id"])

print(df.shape)

In [ ]:
df["content"] = (
    df["artist_name"].astype(str) + " " +
    df["artist_name"].astype(str) + " " +
    df["main_genre"].astype(str) + " " +
    df["composer"].astype(str) + " " +
    df["lyricist"].astype(str) + " " +
    df["language"].astype(str)
)

In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(df["content"])

print(tfidf_matrix.shape)

In [ ]:
knn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=10
)

knn_model.fit(tfidf_matrix)

print("KNN trained.")

In [ ]:
song_idx = pd.Series(
    df.index,
    index=df["song_id"]
).drop_duplicates()

In [ ]:
def recommend_song(song_id, top_n=5):

    if song_id not in song_idx:
        return "Song not found"

    idx = song_idx[song_id]

    distances, indices = knn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 1
    )

    rec_indices = indices.flatten()[1:]

    recommendations = df.iloc[rec_indices][
        [
            "song_id",
            "song_name",
            "artist_name",
            "main_genre"
        ]
    ]

    return recommendations

In [ ]:
test_song = df.iloc[0]["song_id"]

recommend_song(test_song)

In [ ]:
model_package = {
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "knn_model": knn_model,
    "songs_df": df,
    "song_idx": song_idx
}

In [ ]:
with open("content_based_model.pkl", "wb") as f:
    pickle.dump(model_package, f)

print("Model saved.")